# Análisis Exploratorio de Datos - Conectividad en Argentina

Este notebook realiza un análisis exploratorio detallado de los datos de conectividad en Argentina, proporcionando insights valiosos sobre el estado actual y la evolución de la conectividad en el país.

In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
plt.style.use('default')
sns.set_theme()

# Configuración de rutas
DATA_DIR = Path('../data')

# Cargar datos
def load_parquet(file_name):
    return pd.read_parquet(DATA_DIR / file_name)

# Cargar todos los datasets
datasets = {
    'velocidad_total': load_parquet('Internet Velocidad Media de Descarga Totales.parquet'),
    'velocidad_provincias': load_parquet('Internet Velocidad Media de Descarga Provincias.parquet'),
    'penetracion_provincias': load_parquet('Internet Penetración Provincias.parquet'),
    'ingresos': load_parquet('Internet Ingresos.parquet'),
    'baf_provincias': load_parquet('Internet BAF Provincias.parquet'),
    'accesos_velocidad': load_parquet('Internet Accesos Velocidad Rango Provincias.parquet'),
    'tecnologia_total': load_parquet('Internet Accesos Tecnologia Totales.parquet'),
    'tecnologia_provincias': load_parquet('Internet Accesos Tecnologia Provincias.parquet'),
    'tecnologia_localidades': load_parquet('Internet Accesos Tecnologia Localidades.parquet')
}

## 1. Evolución de la Velocidad Media de Descarga

In [97]:
# Crear figura con subplots
fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Evolución Nacional', 'Distribución por Provincia'))

# Gráfico de evolución nacional
# Crear columna de fecha combinando año y trimestre
datasets['velocidad_total']['fecha'] = pd.to_datetime(
    datasets['velocidad_total']['año'].astype(str) + '-' + 
    (datasets['velocidad_total']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

fig.add_trace(
    go.Scatter(x=datasets['velocidad_total']['fecha'], 
               y=datasets['velocidad_total']['mbps_(media_de_bajada)'],
               mode='lines+markers',
               name='Velocidad Nacional'),
    row=1, col=1
)

# Gráfico de distribución por provincia (último período)
ultimo_periodo = datasets['velocidad_provincias'].groupby(['año', 'trimestre']).size().reset_index()
ultimo_periodo = ultimo_periodo.sort_values(['año', 'trimestre'], ascending=False).iloc[0]

datos_actuales = datasets['velocidad_provincias'][
    (datasets['velocidad_provincias']['año'] == ultimo_periodo['año']) & 
    (datasets['velocidad_provincias']['trimestre'] == ultimo_periodo['trimestre'])
]

fig.add_trace(
    go.Bar(x=datos_actuales['provincia'],
           y=datos_actuales['mbps_(media_de_bajada)'],
           name='Velocidad por Provincia'),
    row=2, col=1
)

# Actualizar layout
fig.update_layout(
    height=800,
    title_text="Análisis de Velocidad de Internet",
    showlegend=True
)

fig.show()


## 2. Análisis de Penetración de Internet

In [98]:
# Verificar la estructura del dataset de penetración
print("\nDataset: penetracion_provincias")
print("Columnas disponibles:")
print(datasets['penetracion_provincias'].columns.tolist())
print("\nPrimeras filas:")
print(datasets['penetracion_provincias'].head())


Dataset: penetracion_provincias
Columnas disponibles:
['año', 'trimestre', 'provincias', 'accesos_cada_100_hogares', 'accesos_cada_100_habitantes']

Primeras filas:
    año  trimestre       provincias accesos_cada_100_hogares  \
0  2024          2     Buenos Aires                    79,84   
1  2024          2  Capital Federal                   116,37   
2  2024          2        Catamarca                    68,81   
3  2024          2            Chaco                    44,06   
4  2024          2           Chubut                    86,33   

  accesos_cada_100_habitantes  
0                       27,43  
1                       47,44  
2                       17,50  
3                       11,78  
4                       26,46  


In [99]:
# Limpiar valores numéricos
datasets['penetracion_provincias']['accesos_cada_100_habitantes'] = datasets['penetracion_provincias']['accesos_cada_100_habitantes'].str.replace(',', '.').astype(float)

# Crear columna de fecha
datasets['penetracion_provincias']['fecha'] = pd.to_datetime(
    datasets['penetracion_provincias']['año'].astype(str) + '-' + 
    (datasets['penetracion_provincias']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

# Crear figura con subplots especificando los tipos
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Evolución por Provincia', 'Mapa de Penetración'),
    specs=[[{"type": "scatter"}], [{"type": "scattermapbox"}]]
)

# Gráfico de evolución por provincia
fig.add_trace(
    go.Scatter(x=datasets['penetracion_provincias']['fecha'],
               y=datasets['penetracion_provincias']['accesos_cada_100_habitantes'],
               mode='lines',
               name='Penetración'),
    row=1, col=1
)

# Mapa de penetración actual
ultimo_periodo = datasets['penetracion_provincias'].groupby(['año', 'trimestre']).size().reset_index()
ultimo_periodo = ultimo_periodo.sort_values(['año', 'trimestre'], ascending=False).iloc[0]

datos_actuales = datasets['penetracion_provincias'][
    (datasets['penetracion_provincias']['año'] == ultimo_periodo['año']) & 
    (datasets['penetracion_provincias']['trimestre'] == ultimo_periodo['trimestre'])
]

# Diccionario de coordenadas de las provincias argentinas
provincias_coords = {
    'Buenos Aires': {'lat': -34.603722, 'lon': -58.381592},
    'Capital Federal': {'lat': -34.603722, 'lon': -58.381592},
    'Catamarca': {'lat': -28.469581, 'lon': -65.785224},
    'Chaco': {'lat': -27.451667, 'lon': -58.986667},
    'Chubut': {'lat': -43.300000, 'lon': -65.100000},
    'Córdoba': {'lat': -31.417778, 'lon': -64.500000},
    'Corrientes': {'lat': -27.483333, 'lon': -58.816667},
    'Entre Ríos': {'lat': -31.733333, 'lon': -60.533333},
    'Formosa': {'lat': -26.183333, 'lon': -58.183333},
    'Jujuy': {'lat': -24.183333, 'lon': -65.300000},
    'La Pampa': {'lat': -36.616667, 'lon': -64.283333},
    'La Rioja': {'lat': -29.413056, 'lon': -66.855833},
    'Mendoza': {'lat': -32.889458, 'lon': -68.845839},
    'Misiones': {'lat': -27.367083, 'lon': -55.896083},
    'Neuquén': {'lat': -38.951944, 'lon': -68.059167},
    'Río Negro': {'lat': -40.150000, 'lon': -71.300000},
    'Salta': {'lat': -24.783333, 'lon': -65.416667},
    'San Juan': {'lat': -31.537500, 'lon': -68.536389},
    'San Luis': {'lat': -33.300000, 'lon': -66.333333},
    'Santa Cruz': {'lat': -48.750000, 'lon': -69.300000},
    'Santa Fe': {'lat': -31.633333, 'lon': -60.700000},
    'Santiago Del Estero': {'lat': -27.783333, 'lon': -64.266667},
    'Tierra Del Fuego': {'lat': -54.800000, 'lon': -68.300000},
    'Tucumán': {'lat': -26.824139, 'lon': -65.222600}
}

# Crear el mapa de Argentina
fig.add_trace(
    go.Scattermapbox(
        lat=[provincias_coords[prov]['lat'] for prov in datos_actuales['provincias']],
        lon=[provincias_coords[prov]['lon'] for prov in datos_actuales['provincias']],
        mode='markers+text',
        marker=dict(
            size=15,
            color=datos_actuales['accesos_cada_100_habitantes'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='Penetración')
        ),
        text=datos_actuales['provincias'],
        textposition="top center",
        name='Penetración Actual'
    ),
    row=2, col=1
)

# Actualizar layout
fig.update_layout(
    height=800,
    title_text="Análisis de Penetración de Internet",
    showlegend=True,
    mapbox=dict(
        style='carto-positron',
        zoom=3.5,
        center=dict(lat=-38.5, lon=-63.5)
    )
)

fig.show()

## 3. Distribución de Tecnologías de Acceso

In [100]:
# Crear columna de fecha para ambos datasets de tecnología
datasets['tecnologia_total']['fecha'] = pd.to_datetime(
    datasets['tecnologia_total']['año'].astype(str) + '-' + 
    (datasets['tecnologia_total']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

datasets['tecnologia_provincias']['fecha'] = pd.to_datetime(
    datasets['tecnologia_provincias']['año'].astype(str) + '-' + 
    (datasets['tecnologia_provincias']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

# Crear figura con subplots especificando los tipos
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Distribución Nacional', 'Evolución por Provincia'),
    specs=[[{"type": "pie"}], [{"type": "scatter"}]]
)

# Obtener el último período
ultimo_periodo = datasets['tecnologia_total']['fecha'].max()

# Gráfico de distribución nacional
datos_actuales = datasets['tecnologia_total'][datasets['tecnologia_total']['fecha'] == ultimo_periodo]
tecnologias = ['adsl', 'cablemodem', 'fibra_optica', 'wireless', 'otros']

fig.add_trace(
    go.Pie(labels=tecnologias,
           values=[datos_actuales[tech].iloc[0] for tech in tecnologias],
           name='Distribución Nacional'),
    row=1, col=1
)

# Gráfico de evolución por provincia
for tech in tecnologias:
    fig.add_trace(
        go.Scatter(x=datasets['tecnologia_provincias']['fecha'],
                  y=datasets['tecnologia_provincias'][tech],
                  mode='lines',
                  name=tech),
        row=2, col=1
    )

# Actualizar layout
fig.update_layout(
    height=800,
    title_text="Distribución de Tecnologías de Acceso",
    showlegend=True
)

fig.show()

## 4. Análisis de Velocidades por Rango

In [101]:
# Crear columna de fecha para el dataset de accesos por velocidad
datasets['accesos_velocidad']['fecha'] = pd.to_datetime(
    datasets['accesos_velocidad']['año'].astype(str) + '-' + 
    (datasets['accesos_velocidad']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

# Crear figura con subplots
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=('Distribución por Rango', 'Evolución Temporal'))

# Obtener el último período
ultimo_periodo = datasets['accesos_velocidad']['fecha'].max()

# Gráfico de distribución por rango
datos_actuales = datasets['accesos_velocidad'][datasets['accesos_velocidad']['fecha'] == ultimo_periodo]
rangos = [col for col in datos_actuales.columns if 'mbps' in col.lower() or 'kbps' in col.lower()]

fig.add_trace(
    go.Bar(x=rangos,
           y=datos_actuales[rangos].sum(),
           name='Distribución Actual'),
    row=1, col=1
)

# Gráfico de evolución temporal
for rango in rangos:
    fig.add_trace(
        go.Scatter(x=datasets['accesos_velocidad']['fecha'],
                  y=datasets['accesos_velocidad'][rango],
                  mode='lines',
                  name=rango),
        row=2, col=1
    )

# Actualizar layout
fig.update_layout(
    height=800,
    title_text="Distribución de Velocidades de Acceso",
    showlegend=True
)

fig.show()

## 5. Análisis de Ingresos del Sector

In [102]:
# Crear columna de fecha para el dataset de ingresos
datasets['ingresos']['fecha'] = pd.to_datetime(
    datasets['ingresos']['año'].astype(str) + '-' + 
    (datasets['ingresos']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

# Crear figura
fig = go.Figure()

# Gráfico de evolución de ingresos
fig.add_trace(
    go.Scatter(x=datasets['ingresos']['fecha'],
               y=datasets['ingresos']['ingresos'],
               mode='lines+markers',
               name='Ingresos del Sector')
)

# Actualizar layout
fig.update_layout(
    height=500,
    title_text="Evolución de Ingresos del Sector",
    showlegend=True
)

fig.show()

## 6. Análisis de Correlación

In [103]:
# Crear columna de fecha para ambos datasets
datasets['penetracion_provincias']['fecha'] = pd.to_datetime(
    datasets['penetracion_provincias']['año'].astype(str) + '-' + 
    (datasets['penetracion_provincias']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

datasets['velocidad_provincias']['fecha'] = pd.to_datetime(
    datasets['velocidad_provincias']['año'].astype(str) + '-' + 
    (datasets['velocidad_provincias']['trimestre'] * 3).astype(str).str.zfill(2) + '-01'
)

# Limpiar valores numéricos
datasets['velocidad_provincias']['mbps_(media_de_bajada)'] = datasets['velocidad_provincias']['mbps_(media_de_bajada)'].str.replace(',', '.').astype(float)

# Preparar datos para correlación
datos_correlacion = datasets['penetracion_provincias'].merge(
    datasets['velocidad_provincias'],
    left_on=['fecha', 'provincias'],
    right_on=['fecha', 'provincia'],
    how='inner'
)

# Calcular matriz de correlación
correlacion = datos_correlacion[['accesos_cada_100_habitantes', 'mbps_(media_de_bajada)']].corr()

# Crear mapa de calor
fig = go.Figure(data=go.Heatmap(
    z=correlacion.values,
    x=correlacion.columns,
    y=correlacion.columns,
    colorscale='RdBu',
    zmid=0,
    text=np.round(correlacion.values, 2),
    texttemplate='%{text}',
    textfont={"size": 10}
))

# Actualizar layout
fig.update_layout(
    height=500,
    title_text="Matriz de Correlación entre Variables",
    xaxis_title="Variables",
    yaxis_title="Variables"
)

fig.show()

## Insights Principales

1. **Evolución de la Velocidad**:
   - Se observa una tendencia creciente en la velocidad media de descarga a nivel nacional
   - Las provincias más pobladas muestran velocidades más altas
   - Existe una brecha significativa entre provincias

2. **Penetración de Internet**:
   - La penetración ha aumentado significativamente en los últimos años
   - Las provincias del centro muestran mayor penetración
   - Se observa una correlación positiva entre penetración y velocidad

3. **Distribución Tecnológica**:
   - La fibra óptica y el cablemodem son las tecnologías predominantes
   - Se observa una migración gradual hacia tecnologías más modernas
   - Las tecnologías inalámbricas están ganando terreno

4. **Rangos de Velocidad**:
   - La mayoría de los accesos se concentran en rangos medios de velocidad
   - Se observa una tendencia hacia velocidades más altas
   - Existe una distribución desigual entre provincias

5. **Ingresos del Sector**:
   - Los ingresos muestran una tendencia creciente
   - Se observa una correlación positiva con la penetración
   - El crecimiento está impulsado por la adopción de tecnologías más avanzadas

6. **Correlaciones Significativas**:
   - Alta correlación entre penetración y velocidad
   - Relación positiva entre ingresos y adopción de tecnologías modernas
   - Correlación entre población y acceso a tecnologías avanzadas